### Setup

In [141]:
# Importing libraries

import pandas as pd
import numpy as np
import re
from datetime import datetime
from google_play_scraper import app, reviews, Sort
from langdetect import detect
import os

### Collecting Ids of usecase apps

In [142]:
CBE_BANK_ID = 'com.combanketh.mobilebanking'
BOA_BANK_ID = 'com.boa.boaMobileBanking'
DASHEN_BANK_ID = 'com.dashen.dashensuperapp'

usercase_apps = [CBE_BANK_ID, BOA_BANK_ID, DASHEN_BANK_ID]


#### App info summary

In [143]:
def fetch_app_info(app_id):

    app_info = app(app_id, lang='en', country='et')
    print( "=" * 50 )
    print( "App Information" )
    print( "=" * 50 )

    # printing some key information about the app
    print(f"Title: {app_info['title']}")
    print(f"Developer: {app_info['developer']}")
    print(f"Rating: {app_info['ratings']}")
    print(f"Number of Reviews: {app_info['reviews']}")
    print(f"score: {app_info['score']}")
    print(f"Installs: {app_info['installs']}")
    print(f"reviews: {app_info['reviews']}")
    

In [144]:
for app_id in usercase_apps:
    fetch_app_info(app_id)

App Information
Title: Commercial Bank of Ethiopia
Developer: Commercial Bank Of Ethiopia
Rating: 48341
Number of Reviews: 9310
score: 4.286794
Installs: 5,000,000+
reviews: 9310
App Information
Title: BoA Mobile
Developer: Bank of Abyssinia
Rating: 9224
Number of Reviews: 1461
score: 4.391259
Installs: 1,000,000+
reviews: 1461
App Information
Title: Dashen Bank
Developer: Dashen Bank S.c.
Rating: 5622
Number of Reviews: 1023
score: 4.2255774
Installs: 1,000,000+
reviews: 1023


#### Scraping reviews

In [145]:
def fetch_reviews(app_id, num_reviews=1000):
    result, _ = reviews(
        app_id, 
        lang='en', 
        country='et', 
        sort=Sort.NEWEST, 
        count=num_reviews
        )
    return result

In [146]:
raw_reviews_cbe, raw_reviews_boa, raw_reviews_dashen = [], [], []
for app_id in usercase_apps:
    raw_reviews = fetch_reviews(app_id)
    
    if app_id == CBE_BANK_ID:
        for review in raw_reviews:
            raw_reviews_cbe.append(
                {
                    "review_id": review['reviewId'],
                    "rating": review['score'],
                    "review": review['content'],
                    "date": review['at'],
                    "bank": "CBE",
                    'source': 'Google Play Store'
                }
            )
    elif app_id == BOA_BANK_ID:
        for review in raw_reviews:
            raw_reviews_boa.append(
                {
                    "review_id": review['reviewId'],
                    "rating": review['score'],
                    "review": review['content'],
                    "date": review['at'],
                    "bank": "BOA",
                    'source': 'Google Play Store'
                }
            )
    elif app_id == DASHEN_BANK_ID:
        for review in raw_reviews:
            raw_reviews_dashen.append(
                {
                    "review_id": review['reviewId'],
                    "rating": review['score'],
                    "review": review['content'],
                    "date": review['at'],
                    "bank": "DASHEN",
                    'source': 'Google Play Store'
                }
            )
    

### Converting the data into Dataframe

In [147]:
df_raw_cbe = pd.DataFrame(raw_reviews_cbe)
df_raw_boa = pd.DataFrame(raw_reviews_boa)
df_raw_dashen = pd.DataFrame(raw_reviews_dashen)

### EDA

In [148]:
df_raw_boa.head()

,review_id,rating,review,date,bank,source
0,c3bb042c-844b-4580-98b9-df418622b2fb,5,it's very good app,2026-05-12 11:50:32,BOA,Google Play Store
1,400ce769-3726-43b2-ac4d-755b3a15f026,2,this app is good but the speed of app is very ...,2026-05-11 18:18:54,BOA,Google Play Store
2,4d6d2f22-5e71-47be-9cde-a1cf6c9fff93,5,good,2026-05-09 14:41:50,BOA,Google Play Store
3,e77089b3-aecf-45e2-a64a-ce917fc4233a,5,boa the best,2026-05-08 13:47:07,BOA,Google Play Store
4,41c64c67-b81a-4326-83c4-e95044aef7f6,5,bank of absiniya is best bank in ethiopian,2026-05-07 10:33:06,BOA,Google Play Store


In [149]:
df_raw_cbe.dtypes

review_id               str
rating                int64
review                  str
date         datetime64[us]
bank                    str
source                  str
dtype: object

In [150]:
df_raw_boa.dtypes

review_id               str
rating                int64
review                  str
date         datetime64[us]
bank                    str
source                  str
dtype: object

In [151]:
df_raw_dashen.dtypes

review_id               str
rating                int64
review                  str
date         datetime64[us]
bank                    str
source                  str
dtype: object

In [152]:
# what's the count for each rating?
print("Rating distribution for BOA:")
rating_counts = df_raw_boa['rating'].value_counts()
for rating, count in rating_counts.items():
    bar = '⭐' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")
print("Rating distribution for CBE:")
rating_counts = df_raw_cbe['rating'].value_counts()
for rating, count in rating_counts.items():
    bar = '⭐' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")
print("Rating distribution for Dashen:")
rating_counts = df_raw_dashen['rating'].value_counts()
for rating, count in rating_counts.items():
    bar = '⭐' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")


Rating distribution for BOA:
  5 stars:  476  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  1 stars:  377  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  4 stars:   56  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  3 stars:   56  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  2 stars:   35  ⭐⭐⭐⭐⭐⭐⭐
Rating distribution for CBE:
  5 stars:  666  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  1 stars:  148  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  4 stars:   78  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  3 stars:   64  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  2 stars:   44  ⭐⭐⭐⭐⭐⭐⭐⭐
Rating distribution for Dashen:
  5 stars:  711  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  1 stars:  146  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  4 stars:   61  ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
  3 stars:   43  ⭐⭐⭐⭐⭐⭐⭐⭐
  2 stars:   39  ⭐⭐⭐⭐⭐⭐⭐


### Data Quality Audit
Let's audit our three common data quality problems:
1. Missing values
2. Duplicate reviews  
3. Inconsistent date formats

In [153]:
# 1. Missing Values Check
for df, name in zip([df_raw_cbe, df_raw_boa, df_raw_dashen], ['CBE', 'BOA', 'DASHEN']):
    print(f"\n{name} Reviews - Missing Values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    for col in df.columns:
        status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
        print(f"  {col:<15}: {status}")


CBE Reviews - Missing Values:
  review_id      : OK
  rating         : OK
  review         : OK
  date           : OK
  bank           : OK
  source         : OK

BOA Reviews - Missing Values:
  review_id      : OK
  rating         : OK
  review         : OK
  date           : OK
  bank           : OK
  source         : OK

DASHEN Reviews - Missing Values:
  review_id      : OK
  rating         : OK
  review         : OK
  date           : OK
  bank           : OK
  source         : OK


In [154]:
# 2 . Duplicate Check

for df, name in zip([df_raw_cbe, df_raw_boa, df_raw_dashen], ['CBE', 'BOA', 'DASHEN']):
    print(f"\n{name} Reviews - Duplicate Rows:")
    duplicate_count = df.duplicated().sum()
    print(f"  Duplicate Rows: {duplicate_count} ({(duplicate_count / len(df) * 100).round(2)}%)")


CBE Reviews - Duplicate Rows:
  Duplicate Rows: 0 (0.0%)

BOA Reviews - Duplicate Rows:
  Duplicate Rows: 0 (0.0%)

DASHEN Reviews - Duplicate Rows:
  Duplicate Rows: 0 (0.0%)


In [155]:
for df, name in zip([df_raw_cbe, df_raw_boa, df_raw_dashen], ['CBE', 'BOA', 'DASHEN']):
    print(f"\n{name} Reviews - Duplicate Rows (by review_id):")
    duplicate_rows_by_id = df.duplicated(['review_id'], keep=False).sum()
    print(f"  Duplicate Rows (by review_id): {duplicate_rows_by_id} ({(duplicate_rows_by_id / len(df) * 100).round(2)}%)")


CBE Reviews - Duplicate Rows (by review_id):
  Duplicate Rows (by review_id): 0 (0.0%)

BOA Reviews - Duplicate Rows (by review_id):
  Duplicate Rows (by review_id): 0 (0.0%)

DASHEN Reviews - Duplicate Rows (by review_id):
  Duplicate Rows (by review_id): 0 (0.0%)


In [156]:
# 3. Date Format Check
# Check if 'date' column is in datetime format:  YYYY-MM-DD

for df, name in zip([df_raw_cbe, df_raw_boa, df_raw_dashen], ['CBE', 'BOA', 'DASHEN']):
    date_point = df['date'].iloc[0]
    print(f"\n{name} Reviews - Sample Date Value:")
    print(f"  Sample date: {date_point}")


CBE Reviews - Sample Date Value:
  Sample date: 2026-05-14 12:13:13

BOA Reviews - Sample Date Value:
  Sample date: 2026-05-12 11:50:32

DASHEN Reviews - Sample Date Value:
  Sample date: 2026-05-13 21:37:46


### Cleaning Strategy

Now we fix each problem, one at a time.  
We work on a **copy** so we can always compare back to the raw data.

### Strategy Overview

| Problem | Strategy | Tool |
|---------|----------|------|
| Missing critical data | Drop the row | `df.dropna()` |
| Duplicate reviews | Keep first occurrence | `df.drop_duplicates()` |
| Inconsistent dates | Parse and reformat | `pd.to_datetime()` |
| Messy and non-Englush text | Strip whitespace / remove junk | `str.strip()`, `re.sub()` |

In [157]:
# copying the raw dataframes
df_cbe = df_raw_cbe.copy()
df_boa = df_raw_boa.copy()
df_dashen = df_raw_dashen.copy()

In [158]:
def is_english_text(text):
    """Return True when the text looks like English."""
    if pd.isna(text):
        return False

    text = str(text).strip()
    if not text:
        return False

    try:
        return detect(text) == "en"
    except Exception:
        return text.isascii()


def clean_text(text):
    """Standardize review text and return null for non-English text."""
    if not is_english_text(text):
        return None

    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

#### cleaning the review text


In [159]:
# cleaning the review text

for df in [df_cbe, df_boa, df_dashen]:
    df['review'] = df['review'].apply(clean_text)

In [160]:
# Doing missing value check again after cleaning
for df, name in zip([df_cbe, df_boa, df_dashen], ['CBE', 'BOA', 'DASHEN']):
    print(f"\n{name} Reviews - Missing Values After Cleaning:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    for col in df.columns:
        status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
        print(f"  {col:<15}: {status}")


CBE Reviews - Missing Values After Cleaning:
  review_id      : OK
  rating         : OK
  review         : 496 missing (49.6%)
  date           : OK
  bank           : OK
  source         : OK

BOA Reviews - Missing Values After Cleaning:
  review_id      : OK
  rating         : OK
  review         : 442 missing (44.2%)
  date           : OK
  bank           : OK
  source         : OK

DASHEN Reviews - Missing Values After Cleaning:
  review_id      : OK
  rating         : OK
  review         : 378 missing (37.8%)
  date           : OK
  bank           : OK
  source         : OK


#### Dropping nulled rows

In [161]:
df_cbe.dropna(inplace=True)
df_boa.dropna(inplace=True)
df_dashen.dropna(inplace=True)

#### Dropping duplicate rows

In [162]:
df_cbe.drop_duplicates(inplace=True)
df_boa.drop_duplicates(inplace=True)
df_dashen.drop_duplicates(inplace=True)

#### Validating values of 'rating' column to be int and is between 1 and 5

In [163]:
# validate rating 1 <= rating <= 5
for df in [df_cbe, df_boa, df_dashen]:
    print(f"Shape before filtering: {df.shape}")
    df = df[df['rating'].between(1, 5)]
    print(f"Shape after filtering: {df.shape}")
    df['rating'] = df['rating'].astype(int)

Shape before filtering: (504, 6)
Shape after filtering: (504, 6)
Shape before filtering: (558, 6)
Shape after filtering: (558, 6)
Shape before filtering: (622, 6)
Shape after filtering: (622, 6)


#### Normalize Date

In [164]:
for df in [df_cbe, df_boa, df_dashen]:
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

### Final Output

In [165]:
df_cbe_cleaned = df_cbe.drop(columns=['review_id'])
df_boa_cleaned = df_boa.drop(columns=['review_id'])
df_dashen_cleaned = df_dashen.drop(columns=['review_id'])

In [166]:
df_cbe_cleaned.sort_values('date', ascending=False, inplace=True)
df_boa_cleaned.sort_values('date', ascending=False, inplace=True)
df_dashen_cleaned.sort_values('date', ascending=False, inplace=True)


In [167]:
df_cbe_cleaned.reset_index(drop=True, inplace=True)
df_boa_cleaned.reset_index(drop=True, inplace=True)
df_dashen_cleaned.reset_index(drop=True, inplace=True)

In [168]:
df_cbe_cleaned.head()

,rating,review,date,bank,source
0,5,incredible,2026-05-14,CBE,Google Play Store
1,5,best app for financial sector,2026-05-14,CBE,Google Play Store
2,5,it's a good application,2026-05-13,CBE,Google Play Store
3,5,"Nice, but I can't get some recently transactio...",2026-05-13,CBE,Google Play Store
4,1,Very Secure but very poor interface and limite...,2026-05-13,CBE,Google Play Store


#### Saving to CSV

In [169]:
os.makedirs('../data/processed', exist_ok=True)

for df, name in zip([df_cbe_cleaned, df_boa_cleaned, df_dashen_cleaned], ['CBE', 'BOA', 'DASHEN']):
    output_path = f'../data/processed/{name.lower()}_bank_reviews_clean.csv'
    df.to_csv(output_path, index=False)
    print(f"Saved {name} reviews to: {output_path}")

Saved CBE reviews to: ../data/processed/cbe_bank_reviews_clean.csv
Saved BOA reviews to: ../data/processed/boa_bank_reviews_clean.csv
Saved DASHEN reviews to: ../data/processed/dashen_bank_reviews_clean.csv


### Preprocessing Report

In [171]:
def bank_report(raw_df, clean_df, name, sample_n=5):
    """Print a compact, human-friendly report comparing raw vs cleaned data for one bank."""
    print('\n' + '='*88)
    title = f"{name} Reviews — Raw vs Cleaned Report"
    print(title.center(88))
    print('='*88 + '\n')

    raw_rows, raw_cols = raw_df.shape
    clean_rows, clean_cols = clean_df.shape
    removed = raw_rows - clean_rows
    removed_pct = (removed / raw_rows * 100) if raw_rows else 0

    print(f"Rows: raw = {raw_rows}, cleaned = {clean_rows} (removed {removed} | {removed_pct:.1f}%)\n")

    # Columns and dtypes
    print("Columns (raw -> cleaned):")
    raw_cols_list = list(raw_df.columns)
    clean_cols_list = list(clean_df.columns)
    print(f"  raw   : {raw_cols_list}")
    print(f"  clean : {clean_cols_list}\n")

    # Date dtype
    if 'date' in raw_df.columns or 'date' in clean_df.columns:
        raw_date = raw_df['date'].dtype if 'date' in raw_df.columns else 'n/a'
        clean_date = clean_df['date'].dtype if 'date' in clean_df.columns else 'n/a'
        print(f"Date dtype: raw = {raw_date}   ->   cleaned = {clean_date}\n")

    # Missing values comparison
    print("Missing values (raw -> cleaned):")
    all_cols = list(pd.Index(raw_df.columns).union(clean_df.columns))
    for col in all_cols:
        raw_m = int(raw_df[col].isnull().sum()) if col in raw_df.columns else 'dropped'
        clean_m = int(clean_df[col].isnull().sum()) if col in clean_df.columns else 'dropped'
        print(f"  {col:<12}: {raw_m:>6}  ->  {clean_m}")
    print()

    # Rating distribution
    print("Rating distribution (cleaned):")
    if 'rating' in clean_df.columns:
        rc = clean_df['rating'].value_counts().sort_index()
        for rating, count in rc.items():
            print(f"  {int(rating):>2} star : {count}")
    else:
        print("  (no rating column)")
    print()

    # Review length statistics
    def length_stats(series):
        if series is None or series.empty:
            return None
        s = series.dropna().astype(str).str.len()
        if s.empty:
            return None
        desc = s.describe()
        return dict(count=int(desc['count']), min=int(desc['min']), max=int(desc['max']), mean=float(desc['mean']), median=float(s.median()))

    raw_len = length_stats(raw_df['review'] if 'review' in raw_df.columns else pd.Series(dtype='int'))
    clean_len = length_stats(clean_df['review'] if 'review' in clean_df.columns else pd.Series(dtype='int'))

    print("Review length (chars):")
    print(f"  raw  : {raw_len}")
    print(f"  clean: {clean_len}\n")


    print('\n' + '-'*88 + '\n')

In [172]:
# CBE report
bank_report(df_raw_cbe, df_cbe_cleaned, 'CBE')


                          CBE Reviews — Raw vs Cleaned Report                           

Rows: raw = 1000, cleaned = 504 (removed 496 | 49.6%)

Columns (raw -> cleaned):
  raw   : ['review_id', 'rating', 'review', 'date', 'bank', 'source']
  clean : ['rating', 'review', 'date', 'bank', 'source']

Date dtype: raw = datetime64[us]   ->   cleaned = str

Missing values (raw -> cleaned):
  bank        :      0  ->  0
  date        :      0  ->  0
  rating      :      0  ->  0
  review      :      0  ->  0
  review_id   :      0  ->  dropped
  source      :      0  ->  0

Rating distribution (cleaned):
   1 star : 113
   2 star : 38
   3 star : 43
   4 star : 43
   5 star : 267

Review length (chars):
  raw  : {'count': 1000, 'min': 1, 'max': 500, 'mean': 40.481, 'median': 15.0}
  clean: {'count': 504, 'min': 3, 'max': 500, 'mean': 70.05952380952381, 'median': 38.0}


----------------------------------------------------------------------------------------



In [173]:
# BOA report
bank_report(df_raw_boa, df_boa_cleaned, 'BOA')


                          BOA Reviews — Raw vs Cleaned Report                           

Rows: raw = 1000, cleaned = 558 (removed 442 | 44.2%)

Columns (raw -> cleaned):
  raw   : ['review_id', 'rating', 'review', 'date', 'bank', 'source']
  clean : ['rating', 'review', 'date', 'bank', 'source']

Date dtype: raw = datetime64[us]   ->   cleaned = str

Missing values (raw -> cleaned):
  bank        :      0  ->  0
  date        :      0  ->  0
  rating      :      0  ->  0
  review      :      0  ->  0
  review_id   :      0  ->  dropped
  source      :      0  ->  0

Rating distribution (cleaned):
   1 star : 299
   2 star : 27
   3 star : 32
   4 star : 29
   5 star : 171

Review length (chars):
  raw  : {'count': 1000, 'min': 1, 'max': 500, 'mean': 50.843, 'median': 17.0}
  clean: {'count': 558, 'min': 3, 'max': 500, 'mean': 82.77777777777777, 'median': 44.5}


----------------------------------------------------------------------------------------



In [174]:
# Dashen report
bank_report(df_raw_dashen, df_dashen_cleaned, 'DASHEN')


                         DASHEN Reviews — Raw vs Cleaned Report                         

Rows: raw = 1000, cleaned = 622 (removed 378 | 37.8%)

Columns (raw -> cleaned):
  raw   : ['review_id', 'rating', 'review', 'date', 'bank', 'source']
  clean : ['rating', 'review', 'date', 'bank', 'source']

Date dtype: raw = datetime64[us]   ->   cleaned = str

Missing values (raw -> cleaned):
  bank        :      0  ->  0
  date        :      0  ->  0
  rating      :      0  ->  0
  review      :      0  ->  0
  review_id   :      0  ->  dropped
  source      :      0  ->  0

Rating distribution (cleaned):
   1 star : 110
   2 star : 34
   3 star : 31
   4 star : 33
   5 star : 414

Review length (chars):
  raw  : {'count': 1000, 'min': 1, 'max': 500, 'mean': 63.943, 'median': 27.0}
  clean: {'count': 622, 'min': 3, 'max': 500, 'mean': 95.05144694533762, 'median': 61.0}


----------------------------------------------------------------------------------------

